In [0]:

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

CATALOG = "workspace"
TARGET_SCHEMA = "insurance_lakehouse"

BRONZE_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_policy"
BRONZE_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_claims"
SILVER_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_policy"
SILVER_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_claims"
QUALITY_REPORT_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.quality_report"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

DataFrame[]

In [0]:

policy_bronze_df = spark.table(BRONZE_POLICY_TABLE)
claims_bronze_df = spark.table(BRONZE_CLAIMS_TABLE)
policy_silver_df = spark.table(SILVER_POLICY_TABLE)
claims_silver_df = spark.table(SILVER_CLAIMS_TABLE)

results = []

In [0]:

# Row count checks
policy_bronze_count = policy_bronze_df.count()
claims_bronze_count = claims_bronze_df.count()
policy_silver_count = policy_silver_df.count()
claims_silver_count = claims_silver_df.count()

results.append(Row(
    layer="bronze",
    check_name="policy_row_count_gt_0",
    status="PASS" if policy_bronze_count > 0 else "FAIL",
    metric_value=str(policy_bronze_count),
    details="Bronze policy must have rows"
))

results.append(Row(
    layer="bronze",
    check_name="claims_row_count_gt_0",
    status="PASS" if claims_bronze_count > 0 else "FAIL",
    metric_value=str(claims_bronze_count),
    details="Bronze claims must have rows"
))

results.append(Row(
    layer="silver",
    check_name="policy_row_count_gt_0",
    status="PASS" if policy_silver_count > 0 else "FAIL",
    metric_value=str(policy_silver_count),
    details="Silver policy must have rows"
))

results.append(Row(
    layer="silver",
    check_name="claims_row_count_gt_0",
    status="PASS" if claims_silver_count > 0 else "FAIL",
    metric_value=str(claims_silver_count),
    details="Silver claims must have rows"
))

In [0]:

# Duplicate key checks
policy_dup_count = policy_silver_df.groupBy("policy_id").count().filter("count > 1").count()
claim_dup_count = claims_silver_df.groupBy("claim_id").count().filter("count > 1").count()

results.append(Row(
    layer="silver",
    check_name="policy_id_no_duplicates",
    status="PASS" if policy_dup_count == 0 else "FAIL",
    metric_value=str(policy_dup_count),
    details="Silver policy should have one row per policy_id"
))

results.append(Row(
    layer="silver",
    check_name="claim_id_no_duplicates",
    status="PASS" if claim_dup_count == 0 else "FAIL",
    metric_value=str(claim_dup_count),
    details="Silver claims should have one row per claim_id"
))

In [0]:

# Null key checks
policy_null_key_count = policy_silver_df.filter("policy_id is null").count()
claim_null_key_count = claims_silver_df.filter("claim_id is null or policy_id is null").count()

results.append(Row(
    layer="silver",
    check_name="policy_id_not_null",
    status="PASS" if policy_null_key_count == 0 else "FAIL",
    metric_value=str(policy_null_key_count),
    details="policy_id should not be null in silver_policy"
))

results.append(Row(
    layer="silver",
    check_name="claim_keys_not_null",
    status="PASS" if claim_null_key_count == 0 else "FAIL",
    metric_value=str(claim_null_key_count),
    details="claim_id and policy_id should not be null in silver_claims"
))

In [0]:
orphan_claims_count = (
    claims_silver_df.alias("c")
    .join(policy_silver_df.alias("p"), on="policy_id", how="left_anti")
    .count()
)

results.append(Row(
    layer="silver",
    check_name="claims_have_matching_policy",
    status="PASS" if orphan_claims_count == 0 else "FAIL",
    metric_value=str(orphan_claims_count),
    details="Every claim policy_id should exist in silver_policy"
))

In [0]:

report_df = spark.createDataFrame(results).withColumn("check_ts", current_timestamp())

(
    report_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUALITY_REPORT_TABLE)
)

display(spark.table(QUALITY_REPORT_TABLE))

layer,check_name,status,metric_value,details,check_ts
bronze,policy_row_count_gt_0,PASS,300000,Bronze policy must have rows,2026-03-26T21:55:40.535Z
bronze,claims_row_count_gt_0,PASS,500011,Bronze claims must have rows,2026-03-26T21:55:40.535Z
silver,policy_row_count_gt_0,PASS,300000,Silver policy must have rows,2026-03-26T21:55:40.535Z
silver,claims_row_count_gt_0,PASS,500010,Silver claims must have rows,2026-03-26T21:55:40.535Z
silver,policy_id_no_duplicates,PASS,0,Silver policy should have one row per policy_id,2026-03-26T21:55:40.535Z
silver,claim_id_no_duplicates,PASS,0,Silver claims should have one row per claim_id,2026-03-26T21:55:40.535Z
silver,policy_id_not_null,PASS,0,policy_id should not be null in silver_policy,2026-03-26T21:55:40.535Z
silver,claim_keys_not_null,PASS,0,claim_id and policy_id should not be null in silver_claims,2026-03-26T21:55:40.535Z
silver,claims_have_matching_policy,PASS,0,Every claim policy_id should exist in silver_policy,2026-03-26T21:55:40.535Z


In [0]:
display(spark.sql("""
select *
from workspace.insurance_lakehouse.quality_report
where check_name = 'claims_have_matching_policy'
"""))

layer,check_name,status,metric_value,details,check_ts
silver,claims_have_matching_policy,PASS,0,Every claim policy_id should exist in silver_policy,2026-03-26T21:55:40.535Z
